# Live Data Monitoring

Poll a live SQL Race session and display updating parameter values. This notebook demonstrates the polling pattern for monitoring data as it arrives.

**Prerequisites:** A live session actively receiving data.

In [ ]:
import sys
sys.path.insert(0, '.')
from sqlrace_helpers import init_sqlrace, load_session, list_parameters
import time
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

In [ ]:
SESSION_GUID = "<REPLACE WITH YOUR SESSION GUID>"
POLL_INTERVAL = 1.0  # seconds between polls
MAX_POLLS = 30       # stop after this many polls

sm = init_sqlrace()
client_session, session = load_session(sm, SESSION_GUID)

## Select parameters to monitor

In [ ]:
all_params = list_parameters(session)
monitor_params = all_params[:3]  # Monitor first 3 parameters
print(f"Monitoring: {monitor_params}")

## Polling loop

Each poll reads the latest samples and appends them to a rolling buffer. The display updates in-place.

**Note:** In a live session, `session.EndTime` advances as new data arrives. We read samples near the end to get the latest values.

In [ ]:
history = {p: [] for p in monitor_params}
timestamps = []

print("Starting live polling...")
print(f"Polling every {POLL_INTERVAL}s for up to {MAX_POLLS} iterations.")
print("Interrupt the kernel to stop early.\n")

try:
    for poll in range(MAX_POLLS):
        end_time = session.EndTime
        # Read the last 100ms of data
        window_ns = int(0.1 * 1e9)
        start_time = max(session.StartTime, end_time - window_ns)

        latest = {}
        for param_id in monitor_params:
            pda = session.CreateParameterDataAccess(param_id)
            try:
                samples = pda.GetSamplesBetween(start_time, end_time)
                if samples.SampleCount > 0:
                    val = float(samples.Data(samples.SampleCount - 1))
                    latest[param_id] = val
                    history[param_id].append(val)
                else:
                    latest[param_id] = float('nan')
            finally:
                pda.Dispose()

        timestamps.append(poll * POLL_INTERVAL)

        # Update display
        clear_output(wait=True)
        print(f"Poll {poll + 1}/{MAX_POLLS} | Session end: {end_time} ns")
        print("-" * 50)
        for p, v in latest.items():
            print(f"  {p:30s} = {v:12.4f}")

        time.sleep(POLL_INTERVAL)

except KeyboardInterrupt:
    print("\nPolling stopped by user.")

print(f"\nCollected {len(timestamps)} samples.")

## Plot collected history

In [ ]:
if timestamps:
    df_hist = pd.DataFrame(history, index=timestamps)
    df_hist.index.name = "time (s)"

    fig, axes = plt.subplots(len(monitor_params), 1,
                             figsize=(12, 3 * len(monitor_params)),
                             sharex=True, squeeze=False)
    for i, param in enumerate(monitor_params):
        axes[i, 0].plot(df_hist.index, df_hist[param],
                        marker='o', markersize=3, linewidth=1)
        axes[i, 0].set_ylabel(param)
        axes[i, 0].grid(True, alpha=0.3)

    axes[-1, 0].set_xlabel("Poll time (s)")
    fig.suptitle("Live Monitoring History")
    plt.tight_layout()
    plt.show()
else:
    print("No data collected.")

In [ ]:
client_session.Dispose()
print("Session closed.")